# NBA Draft Success - Exploratory Data Analysis
### Data Sources
- **draft_history** — draft picks from 2000 onwards (round, pick, team, school)
- **draft_combine_stats** — physical & athletic measurements (wingspan, vertical, agility) — 2000 onwards, bench press excluded
- **common_player_info** — career info (seasons played, position, active status)

### Goal
Understand the structure, quality, and distributions of each dataset before building a model to predict draft success.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = r"E:\Github\nba\nba.sqlite"
conn = sqlite3.connect(DB_PATH)
print("Connected to database")

## 1. Draft History

In [ ]:
draft = pd.read_sql_query("SELECT * FROM draft_history WHERE season >= 2000", conn)
print(f"Rows: {len(draft)}")
print(f"Columns: {draft.shape[1]}")
print(f"Year range: {draft['season'].min()} - {draft['season'].max()}")
draft.dtypes

In [ ]:
print("Missing values:")
draft.isnull().sum()

In [ ]:
draft.describe()

In [ ]:
draft.head(10)

## 2. Draft Combine Stats

In [ ]:
combine_cols = [
    'season', 'player_id', 'player_name', 'position',
    'height_wo_shoes', 'weight', 'wingspan', 'standing_reach',
    'standing_vertical_leap', 'max_vertical_leap',
    'lane_agility_time', 'modified_lane_agility_time', 'three_quarter_sprint'
]

combine = pd.read_sql_query(
    f"SELECT {', '.join(combine_cols)} FROM draft_combine_stats WHERE season >= 2000", conn
)
print(f"Rows: {len(combine)}")
print(f"Columns: {combine.shape[1]}")
print(f"Year range: {combine['season'].min()} - {combine['season'].max()}")
combine.dtypes

In [ ]:
print("Missing values (%):")
missing = (combine.isnull().sum() / len(combine) * 100).round(1)
print(missing[missing > 0])

In [ ]:
# Missing values by year for key metrics
key_metrics = ['wingspan', 'standing_vertical_leap', 'max_vertical_leap', 'lane_agility_time', 'three_quarter_sprint']
missing_by_year = combine.groupby('season')[key_metrics].apply(lambda x: x.isnull().mean() * 100).round(1)
missing_by_year.plot(kind='bar', figsize=(14, 5), title='Missing Data % by Year')
plt.ylabel('Missing %')
plt.tight_layout()
plt.show()

In [ ]:
combine.describe()

In [ ]:
combine.head(10)

## 3. Player Career Info

In [ ]:
players = pd.read_sql_query("""
    SELECT person_id, display_first_last, position, height, weight,
           season_exp, from_year, to_year,
           draft_year, draft_round, draft_number,
           country, school, greatest_75_flag
    FROM common_player_info
    WHERE draft_year >= 2000
""", conn)
print(f"Rows: {len(players)}")
print(f"Columns: {players.shape[1]}")
players.dtypes

In [ ]:
print("Missing values (%):")
missing = (players.isnull().sum() / len(players) * 100).round(1)
print(missing[missing > 0])

In [ ]:
players.describe(include='all')

In [ ]:
# Distribution of seasons played
players['season_exp'].dropna().plot(kind='hist', bins=20, figsize=(10, 5), title='Distribution of Seasons Played')
plt.xlabel('Seasons in NBA')
plt.tight_layout()
plt.show()

In [ ]:
players.head(10)